In [ ]:
from enum import unique

import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
from numba.core.cgutils import sizeof
#import pooch
from scipy.sparse import csr_matrix
from scipy.io import mmwrite
from pathlib import Path
import anndata
import copy
from scipy.io import mmread
from scipy.sparse import csr_matrix
import glob
import re
print("scanpy version:", sc.__version__)
print("numpy version:", np.__version__)
import helper.plot_helper  as hp
import matplotlib.pyplot as plt
import scvi  # 变分自编码器框架（scVI/scANVI 模型）
import torch  # 底层深度学习后端 scvi需要用到GPU

In [ ]:
hp.setMyStyle()
os.chdir('/mnt/d/lxk/project/jiangshucai20260506')

In [ ]:
rna = sc.read_h5ad('./h5ad/rna_0.h5ad')

In [ ]:
np.unique(rna.obs['celltype'])

In [ ]:
adata = rna[rna.obs['celltype']== 'Myeloid'].copy()

In [ ]:
sc.experimental.pp.highly_variable_genes(
    adata,
    flavor="pearson_residuals",
    n_top_genes=3000,
    layer="counts",
    batch_key="sample"
)

adata_hvg = adata[:, adata.var["highly_variable"]].copy()

scvi.model.SCVI.setup_anndata(
    adata_hvg,
    layer="counts",
    batch_key="sample"
)

model = scvi.model.SCVI(
    adata_hvg,
    n_layers=2,
    n_latent=30,
    gene_likelihood="nb"
)

model.train()

adata.obsm["X_scVI"] = model.get_latent_representation()

sc.pp.neighbors(
    adata,
    use_rep="X_scVI",
    n_neighbors=15
)

sc.tl.umap(adata)

In [ ]:
for resolution in [0.3,0.5,0.7,0.9,1.1]:
    sc.tl.leiden(adata,
                 key_added='clusters',
                 flavor="igraph",
                 n_iterations=-1,
                 resolution = resolution,
                 )
    sc.pl.umap(
            adata,
            color='clusters',
            legend_loc = 'on data',
            size = 10
        )

In [ ]:
sc.tl.leiden(adata,
             key_added='clusters',
             flavor="igraph",
             n_iterations=-1,
             resolution = 0.5,
             )
sc.pl.umap(
        adata,
        color='clusters',
        legend_loc = 'on data',
        size = 10
    )

In [ ]:
sc.pl.umap(
        adata,
        color='clusters',
        size = 10,
        legend_loc = 'on data',
    )

# 找marker

In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby="clusters",
    method="wilcoxon",
    pts=True,
    use_raw=False,
    layer="normalize" if "normalize" in adata.layers else None
)

from scanpy.get import rank_genes_groups_df

df = rank_genes_groups_df(adata, None)

df["pct_diff"] = df["pct_nz_group"] - df["pct_nz_reference"]

df = df[
    (df["logfoldchanges"] > 0.5) &
    (df["pct_nz_group"] > 0.25) &
    (df["pct_diff"] > 0.15) &
    (df["pvals_adj"] < 0.05)
]

df = df[
    ~df["names"].str.startswith(("MT-", "RPS", "RPL", "HB"))
]

df = df[[
    "group", "names", "scores", "logfoldchanges",
    "pvals_adj", "pct_nz_group", "pct_nz_reference", "pct_diff"
]]

df.to_csv("./export/Myeloid_marker.csv", index=False)

In [ ]:
sc.pl.umap(adata, color=["clusters", "sample", "group"])

In [ ]:


marker_genes_dict = {
    "TREM2+ TAM": ["APOC1", "TREM2", "STAB1", "LTC4S", "PLTP", "CX3CR1", "CEBPA"],
    "Cycling myeloid": ["TOP2A", "TYMS", "STMN1", "HMGB2", "PCNA", "SMC4", "CKS1B"],
    "LYVE1+ resident TAM": ["CD163", "SELENOP", "F13A1", "LYVE1", "MRC1", "SIGLEC1", "KLF4"],
    "SPP1+ lipid TAM": ["GPNMB", "SPP1", "FABP5", "CD36", "LPL", "PLIN2", "LGALS3", "CTSL"],
    "HLA-high APC": ["HLA-DPB1", "HLA-DQB1", "HLA-DQA1", "HLA-DRB5", "CD74", "PLCG2"],
    "MMP9+ osteoclast-like TAM": ["MMP9", "ACP5", "CTSK", "SIGLEC15", "ATP6V0D2", "ANPEP", "DCSTAMP"],
    "FCN1+ monocyte": ["FCN1", "S100A8", "S100A9", "VCAN", "LYZ", "S100A10", "EREG", "AREG"],
    "Inflammatory TAM": ["CCL4L2", "CCL3L1", "CCL4", "CCL3", "IL1B", "TNF", "CCL2", "CH25H"],
    "LRMDA+ migratory TAM": ["LRMDA", "DOCK4", "ARHGAP24", "FRMD4A", "ELMO1", "NAV3", "KCNQ3", "SYNDIG1"],
}

sc.pl.dotplot(
    adata,
    marker_genes_dict,
    groupby="clusters",
    dendrogram=False,
    cmap="YlGnBu", # https://matplotlib.org/stable/users/explain/colors/colormaps.html
    #dot_max=0.6,     # 点最大直径比例（默认 1.0）
    #dot_min=0.1,     # 点最小直径比例（默认 0）
    vmax=5,          # 显示的最大表达值，>3 的都显示为同一深红
    #vmin=0,           # 最小值（默认0）
    #save="_markers.pdf"
    standard_scale="var"
)


In [ ]:
cluster2annotation = {
    "0": "TREM2+ TAM",
    "1": "Cycling myeloid",
    "2": "LYVE1+ resident TAM",
    "3": "SPP1+ lipid TAM",
    "4": "HLA-high APC",
    "5": "MMP9+ osteoclast-like TAM",
    "6": "FCN1+ monocyte",
    "7": "Inflammatory TAM",
    "8": "LRMDA+ migratory TAM",
}

In [ ]:
adata.obs["celltype"] = adata.obs["clusters"].map(cluster2annotation).astype("category")
ordered_groups = list(marker_genes_dict.keys())
adata.obs["celltype"] = (
    adata.obs["celltype"]
    .astype("category")
    .cat.set_categories(ordered_groups, ordered=True)
)


In [ ]:
celltype_colors = {
    "TREM2+ TAM": "#4C5F8F",
    "Cycling myeloid": "#B7794B",
    "LYVE1+ resident TAM": "#6FA083",
    "SPP1+ lipid TAM": "#B95F62",
    "HLA-high APC": "#A66C9A",
    "MMP9+ osteoclast-like TAM": "#5BA8A0",
    "FCN1+ monocyte": "#8DA6BF",
    "Inflammatory TAM": "#D8B56D",
    "LRMDA+ migratory TAM": "#9BAE6A",
}

cats = adata.obs["celltype"].cat.categories
adata.uns["celltype_colors"] = [celltype_colors[x] for x in cats]

In [ ]:
sc.pl.umap(adata,color=["celltype", "sample", "group"])

In [ ]:
adata.write_h5ad('./h5ad/myeloid.h5ad')